# Logistic Regression: From Lines to Probabilities

**Session 1:** how does linear regression find the best line?
**Session 2:** how do we know whether that line is actually good, and whether individual variables matter?
**This session:** what happens when the thing we want to predict isn't a number at all — just a category, like pass or fail?

Let's pick up exactly where we left off last time.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, LogisticRegression

pd.set_option("display.precision", 3)

<details>
<summary>Show code</summary>

```python
recap_df = pd.read_csv("../data/exam_pass_classification.csv")

recap_model = LinearRegression()
recap_model.fit(recap_df[["hours_studied"]], recap_df["passed"])

recap_line_x = np.linspace(0, 9, 100)
recap_line_y = recap_model.coef_[0] * recap_line_x + recap_model.intercept_

recap_figure = go.Figure()
recap_figure.add_trace(
    go.Scatter(x=recap_df["hours_studied"], y=recap_df["passed"], mode="markers",
               marker=dict(size=13, color="#2563eb"), name="actual (0 or 1)")
)
recap_figure.add_trace(
    go.Scatter(x=recap_line_x, y=recap_line_y, mode="lines",
               line=dict(color="#dc2626", width=3), name="linear regression fit")
)
recap_figure.add_hline(y=1, line=dict(color="gray", width=1, dash="dot"))
recap_figure.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"))
recap_figure.update_layout(
    title="Last Session's Cliffhanger",
    xaxis_title="Hours studied", yaxis_title="Passed (0 or 1)",
    template="plotly_white", width=750, height=500,
)
recap_figure.show()

print(f"Predicted value at 8.5 hours: {recap_model.predict(pd.DataFrame({'hours_studied': [8.5]}))[0]:.2f}")
```

</details>

In [2]:
recap_df = pd.read_csv("../data/exam_pass_classification.csv")

recap_model = LinearRegression()
recap_model.fit(recap_df[["hours_studied"]], recap_df["passed"])

recap_line_x = np.linspace(0, 9, 100)
recap_line_y = recap_model.coef_[0] * recap_line_x + recap_model.intercept_

recap_figure = go.Figure()
recap_figure.add_trace(
    go.Scatter(x=recap_df["hours_studied"], y=recap_df["passed"], mode="markers",
               marker=dict(size=13, color="#2563eb"), name="actual (0 or 1)")
)
recap_figure.add_trace(
    go.Scatter(x=recap_line_x, y=recap_line_y, mode="lines",
               line=dict(color="#dc2626", width=3), name="linear regression fit")
)
recap_figure.add_hline(y=1, line=dict(color="gray", width=1, dash="dot"))
recap_figure.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"))
recap_figure.update_layout(
    title="Last Session's Cliffhanger",
    xaxis_title="Hours studied", yaxis_title="Passed (0 or 1)",
    template="plotly_white", width=750, height=500,
)
recap_figure.show()

print(f"Predicted value at 8.5 hours: {recap_model.predict(pd.DataFrame({'hours_studied': [8.5]}))[0]:.2f}")

Predicted value at 8.5 hours: 1.33


A prediction of 1.15 or -0.08 is nonsensical for something that can only ever be 0 or 1. Ordinary linear regression has no way to know it should stay inside that range. We need a different model.

## The Sigmoid Function

Here's the fix: take any real number and squash it into the range (0, 1) using this S-shaped curve.

$$\text{sigmoid}(x) = \frac{1}{1 + e^{-x}}$$

<details>
<summary>Show code</summary>

```python
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

sigmoid_x = np.linspace(-10, 10, 200)
sigmoid_y = sigmoid(sigmoid_x)

sigmoid_figure = go.Figure()
sigmoid_figure.add_trace(
    go.Scatter(x=sigmoid_x, y=sigmoid_y, mode="lines", line=dict(color="#7c3aed", width=3))
)
sigmoid_figure.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"))
sigmoid_figure.add_hline(y=1, line=dict(color="gray", width=1, dash="dot"))
sigmoid_figure.update_layout(
    title="The Sigmoid Function",
    xaxis_title="input (any real number)", yaxis_title="output (always between 0 and 1)",
    template="plotly_white", width=650, height=450,
)
sigmoid_figure.show()
```

</details>

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

sigmoid_x = np.linspace(-10, 10, 200)
sigmoid_y = sigmoid(sigmoid_x)

sigmoid_figure = go.Figure()
sigmoid_figure.add_trace(
    go.Scatter(x=sigmoid_x, y=sigmoid_y, mode="lines", line=dict(color="#7c3aed", width=3))
)
sigmoid_figure.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"))
sigmoid_figure.add_hline(y=1, line=dict(color="gray", width=1, dash="dot"))
sigmoid_figure.update_layout(
    title="The Sigmoid Function",
    xaxis_title="input (any real number)", yaxis_title="output (always between 0 and 1)",
    template="plotly_white", width=650, height=450,
)
sigmoid_figure.show()

```text
very negative input  -> output close to 0
input = 0             -> output = 0.5
very positive input  -> output close to 1
```

No matter how extreme the input, the output can never leave (0, 1). That's exactly the property we were missing.

> ### 🙋 Ask the class
>
> - What happens to the sigmoid's output as the input gets very large? Very negative?

## The Logistic Regression Model

Same weighted sum as before — now squashed through the sigmoid:

$$\hat p = \text{sigmoid}(b + wx)$$

```text
Linear regression:    ŷ = b + wx                    (any real-valued prediction)
Logistic regression:  p̂ = sigmoid(b + wx)            (a probability, 0 to 1)
```

Everything you already know about `b` and `w` still applies — they still combine with the input the same way. The only new step is squashing the result through the sigmoid at the very end.

## Fit With Sklearn

New dataset — more students this time, and the outcomes actually overlap near the boundary (a few students who studied ~5 hours passed, a few didn't). Real data rarely splits perfectly cleanly.

In [4]:
pass_df = pd.read_csv("../data/exam_pass_classification_full.csv")
pass_df.head()

,hours_studied,passed
0,0.5,0
1,0.6,0
2,1.3,0
3,1.5,0
4,2.0,0


<details>
<summary>Show code</summary>

```python
logistic_model = LogisticRegression()
logistic_model.fit(pass_df[["hours_studied"]], pass_df["passed"])

curve_x = np.linspace(0, 10, 200)
curve_y = sigmoid(logistic_model.coef_[0][0] * curve_x + logistic_model.intercept_[0])

fit_figure = go.Figure()
fit_figure.add_trace(
    go.Scatter(x=pass_df["hours_studied"], y=pass_df["passed"], mode="markers",
               marker=dict(size=11, color="#2563eb"), name="actual (0 or 1)")
)
fit_figure.add_trace(
    go.Scatter(x=curve_x, y=curve_y, mode="lines", line=dict(color="#dc2626", width=3), name="fitted sigmoid")
)
fit_figure.add_hline(y=0.5, line=dict(color="#f59e0b", width=2, dash="dash"))
fit_figure.update_layout(
    title="Logistic Regression Fit",
    xaxis_title="Hours studied", yaxis_title="Probability of passing",
    template="plotly_white", width=750, height=500,
)
fit_figure.show()

print(f"coef: {logistic_model.coef_[0][0]:.3f}   intercept: {logistic_model.intercept_[0]:.3f}")
```

</details>

In [5]:
logistic_model = LogisticRegression()
logistic_model.fit(pass_df[["hours_studied"]], pass_df["passed"])

curve_x = np.linspace(0, 10, 200)
curve_y = sigmoid(logistic_model.coef_[0][0] * curve_x + logistic_model.intercept_[0])

fit_figure = go.Figure()
fit_figure.add_trace(
    go.Scatter(x=pass_df["hours_studied"], y=pass_df["passed"], mode="markers",
               marker=dict(size=11, color="#2563eb"), name="actual (0 or 1)")
)
fit_figure.add_trace(
    go.Scatter(x=curve_x, y=curve_y, mode="lines", line=dict(color="#dc2626", width=3), name="fitted sigmoid")
)
fit_figure.add_hline(y=0.5, line=dict(color="#f59e0b", width=2, dash="dash"))
fit_figure.update_layout(
    title="Logistic Regression Fit",
    xaxis_title="Hours studied", yaxis_title="Probability of passing",
    template="plotly_white", width=750, height=500,
)
fit_figure.show()

print(f"coef: {logistic_model.coef_[0][0]:.3f}   intercept: {logistic_model.intercept_[0]:.3f}")

coef: 1.050   intercept: -4.958


Unlike the straight line from Section 1, this curve never leaves (0, 1) — no matter how many or how few hours we plug in.

## Predicted Probability vs. Predicted Class

Logistic regression actually gives you two different things, and it's important to keep them straight.

<details>
<summary>Show code</summary>

```python
example_hours = pd.DataFrame({"hours_studied": [3.0, 5.0, 7.0]})

probabilities = logistic_model.predict_proba(example_hours)
classes = logistic_model.predict(example_hours)

for hours, prob, cls in zip(example_hours["hours_studied"], probabilities, classes):
    print(f"{hours} hours -> P(fail)={prob[0]:.3f}, P(pass)={prob[1]:.3f} -> predicted class: {cls}")
```

</details>

In [6]:
example_hours = pd.DataFrame({"hours_studied": [3.0, 5.0, 7.0]})

probabilities = logistic_model.predict_proba(example_hours)
classes = logistic_model.predict(example_hours)

for hours, prob, cls in zip(example_hours["hours_studied"], probabilities, classes):
    print(f"{hours} hours -> P(fail)={prob[0]:.3f}, P(pass)={prob[1]:.3f} -> predicted class: {cls}")

3.0 hours -> P(fail)=0.859, P(pass)=0.141 -> predicted class: 0
5.0 hours -> P(fail)=0.427, P(pass)=0.573 -> predicted class: 1
7.0 hours -> P(fail)=0.084, P(pass)=0.916 -> predicted class: 1


```text
model.predict_proba(...)  -> the actual probability the model computed (two columns: P(0), P(1))
model.predict(...)        -> a single class label, after applying a decision threshold
```

`predict()` is just `predict_proba()` with a cutoff applied — which brings us to the next question: where should that cutoff be?

## The Decision Threshold

By default, sklearn calls anything with predicted probability ≥ 0.5 "pass" and anything below "fail." But 0.5 is a choice, not a law — exactly like the `p < 0.05` convention from last session.

<details>
<summary>Show code</summary>

```python
threshold_demo_hours = np.array([4.0, 4.5, 4.72, 5.0, 5.5])
threshold_demo_probs = sigmoid(logistic_model.coef_[0][0] * threshold_demo_hours + logistic_model.intercept_[0])

for hours, prob in zip(threshold_demo_hours, threshold_demo_probs):
    classification_at_50 = "pass" if prob >= 0.5 else "fail"
    classification_at_70 = "pass" if prob >= 0.7 else "fail"
    print(f"{hours} hours -> p={prob:.3f}  |  at threshold 0.5: {classification_at_50}  |  at threshold 0.7: {classification_at_70}")
```

</details>

In [7]:
threshold_demo_hours = np.array([4.0, 4.5, 4.72, 5.0, 5.5])
threshold_demo_probs = sigmoid(logistic_model.coef_[0][0] * threshold_demo_hours + logistic_model.intercept_[0])

for hours, prob in zip(threshold_demo_hours, threshold_demo_probs):
    classification_at_50 = "pass" if prob >= 0.5 else "fail"
    classification_at_70 = "pass" if prob >= 0.7 else "fail"
    print(f"{hours} hours -> p={prob:.3f}  |  at threshold 0.5: {classification_at_50}  |  at threshold 0.7: {classification_at_70}")

4.0 hours -> p=0.319  |  at threshold 0.5: fail  |  at threshold 0.7: fail
4.5 hours -> p=0.442  |  at threshold 0.5: fail  |  at threshold 0.7: fail
4.72 hours -> p=0.500  |  at threshold 0.5: fail  |  at threshold 0.7: fail
5.0 hours -> p=0.573  |  at threshold 0.5: pass  |  at threshold 0.7: fail
5.5 hours -> p=0.694  |  at threshold 0.5: pass  |  at threshold 0.7: fail


Notice a few students flip from "pass" to "fail" just by moving the threshold from 0.5 to 0.7, with no change to the model itself. The model always outputs a probability — where you draw the line is a separate decision, often driven by the real-world cost of each type of mistake.

> ### 🙋 Ask the class
>
> - If missing a truly at-risk student were very costly, would you want a higher or lower threshold?

> ### 🧑‍🏫 Instructor note
>
> This is a good moment to mention (without deriving) that threshold choice is exactly where precision/recall
> tradeoffs live -- future material, not today's focus.

## Evaluating Classification

For regression we had SSE and R². For classification, the simplest starting point is a **confusion matrix**: how many predictions of each type were right or wrong.

<details>
<summary>Show code</summary>

```python
predicted_classes = logistic_model.predict(pass_df[["hours_studied"]])
actual_classes = pass_df["passed"]

true_positive = int(((predicted_classes == 1) & (actual_classes == 1)).sum())
true_negative = int(((predicted_classes == 0) & (actual_classes == 0)).sum())
false_positive = int(((predicted_classes == 1) & (actual_classes == 0)).sum())
false_negative = int(((predicted_classes == 0) & (actual_classes == 1)).sum())

confusion_table = pd.DataFrame(
    {"Predicted: fail": [true_negative, false_negative], "Predicted: pass": [false_positive, true_positive]},
    index=["Actual: fail", "Actual: pass"],
)
accuracy = (true_positive + true_negative) / len(actual_classes)

print(confusion_table)
print(f"\nAccuracy: {accuracy:.3f}")
```

</details>

In [8]:
predicted_classes = logistic_model.predict(pass_df[["hours_studied"]])
actual_classes = pass_df["passed"]

true_positive = int(((predicted_classes == 1) & (actual_classes == 1)).sum())
true_negative = int(((predicted_classes == 0) & (actual_classes == 0)).sum())
false_positive = int(((predicted_classes == 1) & (actual_classes == 0)).sum())
false_negative = int(((predicted_classes == 0) & (actual_classes == 1)).sum())

confusion_table = pd.DataFrame(
    {"Predicted: fail": [true_negative, false_negative], "Predicted: pass": [false_positive, true_positive]},
    index=["Actual: fail", "Actual: pass"],
)
accuracy = (true_positive + true_negative) / len(actual_classes)

print(confusion_table)
print(f"\nAccuracy: {accuracy:.3f}")

              Predicted: fail  Predicted: pass
Actual: fail               18                3
Actual: pass                3               21

Accuracy: 0.867


```text
accuracy = (correct predictions) / (total predictions)
```

That's intentionally as far as we'll go today — precision, recall, and ROC-AUC are useful follow-up material once accuracy alone starts to feel too coarse (for example, when one class is much rarer than the other), but they're not needed to understand the core model.

## Two Features → a Decision Boundary

Just like Session 1 went from a line to a plane with two features, logistic regression goes from a curve to a **decision boundary** — the line in feature space where predicted probability crosses 0.5.

In [9]:
pass_2d_df = pd.read_csv("../data/exam_pass_2d.csv")
pass_2d_df.head()

,hours_studied,practice_problems,passed
0,6.7,12,1
1,6.3,6,1
2,1.7,7,0
3,1.5,8,0
4,6.4,3,1


<details>
<summary>Show code</summary>

```python
logistic_model_2d = LogisticRegression()
logistic_model_2d.fit(pass_2d_df[["hours_studied", "practice_problems"]], pass_2d_df["passed"])

w1, w2 = logistic_model_2d.coef_[0]
b = logistic_model_2d.intercept_[0]

# decision boundary: b + w1*x1 + w2*x2 = 0  ->  x2 = -(b + w1*x1) / w2
boundary_x1 = np.linspace(pass_2d_df["hours_studied"].min(), pass_2d_df["hours_studied"].max(), 50)
boundary_x2 = -(b + w1 * boundary_x1) / w2

boundary_figure = go.Figure()
for passed_value, color, label in [(0, "#dc2626", "failed"), (1, "#16a34a", "passed")]:
    subset = pass_2d_df[pass_2d_df["passed"] == passed_value]
    boundary_figure.add_trace(
        go.Scatter(x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
                   marker=dict(size=10, color=color), name=label)
    )
boundary_figure.add_trace(
    go.Scatter(x=boundary_x1, y=boundary_x2, mode="lines",
               line=dict(color="#7c3aed", width=3, dash="dash"), name="decision boundary (p = 0.5)")
)
boundary_figure.update_layout(
    title="Decision Boundary: Two Features",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
boundary_figure.show()
```

</details>

In [10]:
logistic_model_2d = LogisticRegression()
logistic_model_2d.fit(pass_2d_df[["hours_studied", "practice_problems"]], pass_2d_df["passed"])

w1, w2 = logistic_model_2d.coef_[0]
b = logistic_model_2d.intercept_[0]

# decision boundary: b + w1*x1 + w2*x2 = 0  ->  x2 = -(b + w1*x1) / w2
boundary_x1 = np.linspace(pass_2d_df["hours_studied"].min(), pass_2d_df["hours_studied"].max(), 50)
boundary_x2 = -(b + w1 * boundary_x1) / w2

boundary_figure = go.Figure()
for passed_value, color, label in [(0, "#dc2626", "failed"), (1, "#16a34a", "passed")]:
    subset = pass_2d_df[pass_2d_df["passed"] == passed_value]
    boundary_figure.add_trace(
        go.Scatter(x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
                   marker=dict(size=10, color=color), name=label)
    )
boundary_figure.add_trace(
    go.Scatter(x=boundary_x1, y=boundary_x2, mode="lines",
               line=dict(color="#7c3aed", width=3, dash="dash"), name="decision boundary (p = 0.5)")
)
boundary_figure.update_layout(
    title="Decision Boundary: Two Features",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
boundary_figure.show()

Everything on one side of the dashed line is predicted "pass," everything on the other side "fail" — with some unavoidable overlap right near the boundary, same as in the 1-feature case.

> ### 🙋 Ask the class
>
> - What do you think happens to this boundary with three features? (Hint: think back to the line-plane-hyperplane progression from Session 1.)

> ### 🧑‍🏫 Instructor note
>
> Answer: with three features the boundary becomes a plane, and beyond that a hyperplane -- exactly the same
> generalization pattern taught in Session 1 Section 20, just now separating classes instead of fitting values.

## Reading the Coefficients

We can look at the sign and rough size of each coefficient without deriving the full log-odds algebra.

In [11]:
coef_table = pd.DataFrame({
    "feature": ["hours_studied", "practice_problems"],
    "coefficient": logistic_model_2d.coef_[0],
})
coef_table

,feature,coefficient
0,hours_studied,0.918
1,practice_problems,0.154


```text
positive coefficient  -> more of this feature increases the predicted probability of passing
negative coefficient  -> more of this feature decreases the predicted probability of passing
larger magnitude      -> a bigger swing in probability per unit change
```

That's as far as we'll take the interpretation today — the precise "log-odds" reading of these numbers is useful future material, not required to use the model correctly.

## Recap

```text
features
   ↓
weighted sum (b + w1*x1 + w2*x2 + ...)
   ↓
sigmoid
   ↓
probability (0 to 1)
   ↓
threshold
   ↓
predicted class
```

Same weighted-sum idea as every session so far — the only new ingredient is the sigmoid at the end, which is exactly what turns an unbounded number into a valid probability.

> Linear regression predicts a number. Logistic regression predicts a probability, and a probability plus a threshold gives you a class.